<a href="https://colab.research.google.com/github/GajanandImmannavar/Deforestation-Detection-using-Satellite-Imagery/blob/main/Deforestation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Tracking Deforestation using high-resolution satellite images with Deep-learning Techniques combining UNet and ResNet





In [ ]:
!pip install segmentation-models-pytorch
import os, glob, cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
import segmentation_models_pytorch as smp
from sklearn.metrics import confusion_matrix

DATASET_DIR = "/content/drive/MyDrive/dataset/Dataset-20260312T163230Z-3-001/Dataset"
RESULTS_DIR = "/content/drive/MyDrive/dataset/Results_Segmentation-20260312T163409Z-3-001/Results_Segmentation"
os.makedirs(RESULTS_DIR, exist_ok=True)

RAW_FOLDER_NAME  = "Raw"
MASK_FOLDER_NAME = "Mask_Gray_From_Raw"
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png")

PATCH_SIZE   = 128
BATCH_SIZE   = 8
NUM_CLASSES  = 5
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
LR           = 1e-3
EPOCHS       = 100

CLASS_NAMES  = ["Forest","Agriculture","Urban","Water","Barren"]
CLASS_COLORS = {
    0:(0,255,0),
    1:(255,255,0),
    2:(128,128,128),
    3:(0,0,255),
    4:(139,69,19)
}

# Collect image-mask pairs

img_paths, mask_paths = [], []

for year_folder in sorted(os.listdir(DATASET_DIR)):
    year_path = os.path.join(DATASET_DIR, year_folder)
    if not os.path.isdir(year_path): continue
    raw_dir  = os.path.join(year_path, RAW_FOLDER_NAME)
    mask_dir = os.path.join(year_path, MASK_FOLDER_NAME)

    imgs  = sorted([f for f in glob.glob(os.path.join(raw_dir, "*.*")) if f.lower().endswith(IMG_EXTENSIONS)])
    masks = sorted([f for f in glob.glob(os.path.join(mask_dir, "*.*")) if f.lower().endswith(IMG_EXTENSIONS)])

    mask_dict = {os.path.basename(m): m for m in masks}
    for img in imgs:
        fname = os.path.basename(img)
        if fname in mask_dict:
            img_paths.append(img)
            mask_paths.append(mask_dict[fname])

print(f"Total matched image-mask pairs: {len(img_paths)}")

# Visualization

def visualize(raw_path, mask_path):
    img = cv2.imread(raw_path); img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    overlay = np.zeros_like(img)
    for class_id, color in CLASS_COLORS.items():
        overlay[mask==class_id] = color
    blended = cv2.addWeighted(img, 0.7, overlay, 0.3, 0)
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1); plt.imshow(img); plt.title("Raw Image"); plt.axis('off')
    plt.subplot(1,3,2); plt.imshow(mask, cmap='gray'); plt.title("Mask"); plt.axis('off')
    plt.subplot(1,3,3); plt.imshow(blended); plt.title("Overlay"); plt.axis('off')
    plt.show()


class LandUseDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transform=None, num_classes=NUM_CLASSES, ignore_index=255):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.num_classes = num_classes
        self.ignore_index = ignore_index

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # Load image
        img = cv2.imread(self.img_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0

        # Load mask
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE).astype(np.int64)

        mask[mask < 0] = self.ignore_index
        mask[mask >= self.num_classes] = self.ignore_index

        # Apply augmentation
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img, mask = augmented['image'], augmented['mask']

        # HWC -> CHW
        img = np.transpose(img, (2,0,1))

        return torch.from_numpy(img).float(), torch.from_numpy(mask).long()

# Augmentations
train_transform = A.Compose([
    A.RandomCrop(PATCH_SIZE, PATCH_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3)
])
val_transform = A.Compose([A.CenterCrop(PATCH_SIZE, PATCH_SIZE)])



split_ratio = 0.8
split = int(len(img_paths) * split_ratio)

train_ds = LandUseDataset(img_paths[:split], mask_paths[:split], transform=train_transform)
val_ds   = LandUseDataset(img_paths[split:], mask_paths[split:], transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=1, num_workers=2, pin_memory=True)



model = smp.Unet("resnet50", encoder_weights="imagenet", in_channels=3, classes=NUM_CLASSES).to(DEVICE)
loss_fn = nn.CrossEntropyLoss(ignore_index=255)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)



def evaluate_segmentation(y_true, y_pred, num_classes):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    iou = []; dice = []
    for i in range(num_classes):
        tp = cm[i,i]; fn = cm[i,:].sum()-tp; fp = cm[:,i].sum()-tp
        iou.append(tp/(tp+fp+fn+1e-6))
        dice.append(2*tp/(2*tp+fp+fn+1e-6))
    mean_iou = np.mean(iou); mean_dice = np.mean(dice)
    return cm, iou, dice, mean_iou, mean_dice



train_loss_hist, val_loss_hist = [], []
mean_iou_hist, mean_dice_hist = [], []
train_acc_hist, val_acc_hist = [], []
best_iou = 0.0
save_path = os.path.join(RESULTS_DIR, "best_model.pth")

for epoch in range(EPOCHS):
    model.train(); train_loss = 0.0
    correct_pixels, total_pixels = 0, 0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()


        preds = torch.argmax(out, 1)
        mask_valid = (y != 255)
        correct_pixels += (preds[mask_valid] == y[mask_valid]).sum().item()
        total_pixels += mask_valid.sum().item()

    train_loss /= len(train_loader)
    train_acc = correct_pixels / total_pixels
    train_acc_hist.append(train_acc)


    model.eval(); val_loss = 0.0
    preds_all, labels_all = [], []
    correct_val, total_val = 0, 0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            val_loss += loss_fn(out, y).item()
            preds = torch.argmax(out, 1)


            mask_valid = (y != 255)
            correct_val += (preds[mask_valid] == y[mask_valid]).sum().item()
            total_val += mask_valid.sum().item()

            preds_all.append(preds.cpu().numpy())
            labels_all.append(y.cpu().numpy())

    val_loss /= len(val_loader)
    val_acc = correct_val / total_val
    val_acc_hist.append(val_acc)

    preds_all = np.concatenate(preds_all)
    labels_all = np.concatenate(labels_all)
    cm, iou, dice, mean_iou, mean_dice = evaluate_segmentation(labels_all.flatten(), preds_all.flatten(), NUM_CLASSES)

    train_loss_hist.append(train_loss)
    val_loss_hist.append(val_loss)
    mean_iou_hist.append(mean_iou)
    mean_dice_hist.append(mean_dice)

    if mean_iou > best_iou:
        best_iou = mean_iou
        torch.save(model.state_dict(), save_path)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss={train_loss:.4f} | Val Loss={val_loss:.4f} | "
          f"Train Acc={train_acc*100:.2f}% | Val Acc={val_acc*100:.2f}% | "
          f"Mean IoU={mean_iou:.3f} | Mean Dice={mean_dice:.3f}")


plt.figure(figsize=(16,5))
plt.subplot(1,3,1)
plt.plot(train_loss_hist,label='Train Loss'); plt.plot(val_loss_hist,label='Val Loss')
plt.title("Loss Curve"); plt.legend(); plt.grid(True)

plt.subplot(1,3,2)
plt.plot(mean_iou_hist,label='Mean IoU'); plt.plot(mean_dice_hist,label='Mean Dice')
plt.title("IoU & Dice Curve"); plt.legend(); plt.grid(True)

plt.subplot(1,3,3)
plt.plot(train_acc_hist,label='Train Accuracy'); plt.plot(val_acc_hist,label='Val Accuracy')
plt.title("Accuracy Curve"); plt.legend(); plt.grid(True)

plt.savefig(os.path.join(RESULTS_DIR,"training_curves.png"))
plt.show()


overall_acc = val_acc_hist[-1]
print(f"Overall Validation Accuracy: {overall_acc*100:.2f}%")


plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix')
plt.savefig(os.path.join(RESULTS_DIR,"confusion_matrix.png"))
plt.show()

metrics_df = pd.DataFrame({"Class":CLASS_NAMES, "IoU":iou, "Dice":dice})
metrics_df.to_csv(os.path.join(RESULTS_DIR,"per_class_metrics.csv"), index=False)
print("Saved per-class metrics")



In [ ]:
# ===== IMPORTS (IMPORTANT AFTER RECONNECT) =====
import os, glob, cv2
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from google.colab import files
from torchvision import models, transforms
from sklearn.metrics.pairwise import cosine_similarity

# ===== GLOBAL VARIABLES (re-defined for robustness) =====
DATASET_DIR = "/content/drive/MyDrive/dataset/Dataset-20260312T163230Z-3-001/Dataset"
RESULTS_DIR = "/content/drive/MyDrive/dataset/Results_Segmentation-20260312T163409Z-3-001/Results_Segmentation"
RAW_FOLDER_NAME  = "Raw"
MASK_FOLDER_NAME = "Mask_Gray_From_Raw"
NUM_CLASSES  = 5
PATCH_SIZE   = 128

CLASS_NAMES  = ["Forest","Agriculture","Urban","Water","Barren"]
CLASS_COLORS = {
    0:(0,255,0),
    1:(255,255,0),
    2:(128,128,128),
    3:(0,0,255),
    4:(139,69,19)
}

# ===== CPU DEVICE =====
DEVICE = "cpu"

# ===== UPLOAD IMAGE =====
uploaded = files.upload()
uploaded_img_path = list(uploaded.keys())[0]
print("Uploaded image:", uploaded_img_path)

# ===== LOAD TRAINED SEGMENTATION MODEL (CPU SAFE) =====
# model needs to be defined if not in global scope
import segmentation_models_pytorch as smp
model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=NUM_CLASSES
).to(DEVICE)

# save_path is already defined in the kernel state
model.load_state_dict(
    torch.load(save_path, map_location=torch.device('cpu'))
)
model.to(DEVICE)
model.eval()

# ===== FEATURE EXTRACTOR (ResNet50) =====
feature_extractor = models.resnet50(pretrained=True)
feature_extractor.fc = nn.Identity()   # remove classifier
feature_extractor = feature_extractor.to(DEVICE)
feature_extractor.eval()

# ===== IMAGE TRANSFORM =====
img_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# ===== FEATURE EXTRACTION =====
def extract_features(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img_tf(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        feat = feature_extractor(img)

    return feat.cpu().numpy().flatten()

# ===== SIMILAR IMAGE SEARCH =====
def find_similar_image(uploaded_img_path, year):
    base_feat = extract_features(uploaded_img_path)

    year_imgs = glob.glob(
        os.path.join(DATASET_DIR, year, RAW_FOLDER_NAME, "*.*")
    )

    best_img = None
    best_score = -1

    for img_path in year_imgs:
        feat = extract_features(img_path)
        score = cosine_similarity([base_feat], [feat])[0][0]

        if score > best_score:
            best_score = score
            best_img = img_path

    return best_img, best_score

# ===== FOREST PROBABILITY MAP =====
def forest_probability_map(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (PATCH_SIZE, PATCH_SIZE))
    img = img.astype(np.float32) / 255.0
    img = np.transpose(img, (2,0,1))

    img = torch.from_numpy(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        out = model(img)
        forest_prob = torch.softmax(out, dim=1)[0,0].cpu().numpy()

    return forest_prob

# ===== PROCESS ALL YEARS =====
years = sorted([
    y for y in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, y))
])

uploaded_forest = forest_probability_map(uploaded_img_path)

for year in years:

    sim_img, sim_score = find_similar_image(uploaded_img_path, year)

    print(f"{year} → matched image: {os.path.basename(sim_img)} | similarity: {sim_score:.3f}")

    year_forest = forest_probability_map(sim_img)

    forest_change = uploaded_forest - year_forest

    plt.figure(figsize=(6,6))
    plt.imshow(forest_change, cmap="RdYlGn")
    plt.colorbar(label="Forest Probability Change")
    plt.title(f"Forest Change (Uploaded → {year})")
    plt.axis("off")

    plt.savefig(os.path.join(
        RESULTS_DIR,
        f"forest_change_uploaded_{year}.png"
    ))

    plt.show()

LAND-USE PURPOSE analysis

In [ ]:
# ===== IMPORTS =====
import os, glob, cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from google.colab import files
from torchvision import models, transforms
from sklearn.metrics.pairwise import cosine_similarity

# ===== DEVICE =====
DEVICE = "cpu"

# ===== UPLOAD IMAGE =====
uploaded = files.upload()
uploaded_img_path = list(uploaded.keys())[0]
print("Uploaded image:", uploaded_img_path)

# ===== LOAD MODEL (CPU SAFE) =====
model.load_state_dict(
    torch.load(save_path, map_location=torch.device('cpu'))
)
model.to(DEVICE)
model.eval()

# ===== LAND-USE PREDICTION =====
def landuse_prediction(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (PATCH_SIZE, PATCH_SIZE))
    img = img.astype(np.float32) / 255.0
    img = np.transpose(img, (2,0,1))
    img = torch.from_numpy(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        out = model(img)
        pred = torch.argmax(out, 1).cpu().numpy()[0]

    return pred

# ===== LAND-USE PERCENTAGE =====
def landuse_percentage(pred):
    total_pixels = pred.size
    percentages = {}

    for i, cls in enumerate(CLASS_NAMES):
        percentages[cls] = (np.sum(pred == i) / total_pixels) * 100

    return percentages

# ===== FEATURE EXTRACTOR =====
feature_extractor = models.resnet50(pretrained=True)
feature_extractor.fc = nn.Identity()
feature_extractor = feature_extractor.to(DEVICE)
feature_extractor.eval()

img_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

def extract_features(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img_tf(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        feat = feature_extractor(img)

    return feat.cpu().numpy().flatten()

# ===== SIMILAR IMAGE SEARCH =====
def find_similar_image(uploaded_img, year):
    base_feat = extract_features(uploaded_img)

    imgs = glob.glob(
        os.path.join(DATASET_DIR, year, RAW_FOLDER_NAME, "*.*")
    )

    best_img, best_score = None, -1

    for img in imgs:
        score = cosine_similarity(
            [base_feat],
            [extract_features(img)]
        )[0][0]

        if score > best_score:
            best_img, best_score = img, score

    return best_img

# ===== PROCESS YEARS =====
years = sorted(os.listdir(DATASET_DIR))
landuse_report = []

uploaded_pred = landuse_prediction(uploaded_img_path)

for year in years:
    sim_img = find_similar_image(uploaded_img_path, year)

    year_pred = landuse_prediction(sim_img)
    year_usage = landuse_percentage(year_pred)

    record = {"Year": year}
    record.update(year_usage)
    landuse_report.append(record)

    print(f"\nLand-Use Purpose in {year}:")
    for k, v in year_usage.items():
        print(f"{k}: {v:.2f}%")

# ===== SAVE REPORT =====
df = pd.DataFrame(landuse_report)
df.to_csv(
    os.path.join(RESULTS_DIR, "landuse_yearwise.csv"),
    index=False
)

# ===== PLOT =====
plt.figure(figsize=(10,6))

for cls in CLASS_NAMES:
    plt.plot(df["Year"], df[cls], marker='o', label=cls)

plt.xlabel("Year")
plt.ylabel("Land-Use Percentage (%)")
plt.title("Year-wise Land-Use Change")
plt.legend()
plt.grid(True)

plt.savefig(
    os.path.join(RESULTS_DIR, "landuse_trend.png")
)

plt.show()